# Sprint 2 — MVP Analítico: Modelo Baseline de Classificação de Risco de Surto

**Como executar de verdade (passo a passo):**
1. Acesse [colab.research.google.com](https://colab.research.google.com/)
   logado com sua conta Google.
2. Menu **Arquivo > Fazer upload de notebook** (*File > Upload notebook*) e
   selecione este arquivo.
3. Menu **Ambiente de execução > Executar tudo**.
4. Aguarde — a execução completa leva **~15 a 25 minutos**
5. Ao final, baixe o modelo gerado (`/content/modelo_baseline_risco_dengue.pkl`).

**Operações realizadas pelo notebook**
1. Verifica se está rodando no Google Colab
2. Coleta os dados brutos direto das fontes públicas: InfoDengue, IBGE
   (população e área), INMET (precipitação) e ANA (saneamento).
3. Integra tudo em uma única base analítica por município × mês.
4. Treina e compara 5 modelos de classificação supervisionada para prever risco de surto epidêmico de
   dengue.
5. Avalia o melhor modelo em um recorte temporal nunca visto no treino.
6. Exporta o modelo final via, pronto para ser baixado.

**Tempo esperado de execução: ~15-25 minutos**, por causa da
coleta do InfoDengue e do download dos zips do INMET.

**Pré-requisito:** apenas autenticação normal do Google Colab

In [ ]:
%pip install -q requests pandas numpy scikit-learn joblib
print("Dependências instaladas/verificadas: requests, pandas, numpy, scikit-learn, joblib")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import requests
import time

from datetime import datetime

print("Bibliotecas importadas: pandas, numpy, matplotlib, requests, time, datetime")
import os

In [ ]:
try:
    import google.colab
    EM_COLAB = "COLAB_RELEASE_TAG" in os.environ
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    print("Ambiente Google Colab detectado — prosseguindo.")
else:
    print(
        "[AVISO] Ambiente Google Colab NÃO detectado"
    )

## 0.1 Configuração central de variáveis

Todas as variáveis usadas no restante do notebook (UF, período,
endpoints, caminhos de arquivo, parâmetros de requisição) são
definidas uma única vez aqui. O resto do notebook só referencia essas
constantes.

In [ ]:
# --- Escopo geográfico ---
UF_SIGLA = "SC"
UF_CODIGO_IBGE = 42

# --- Escopo temporal ---
EY_INICIO = 2015
EY_FIM = 2025

# --- Doença analisada ---
DOENCA = "dengue"

# --- Diretórios de dados ---
DIR_RAW = "/content/data/raw"
DIR_PROCESSED = "/content/data/processed"

# --- Endpoints e identificadores das APIs ---
URL_IBGE_LOCALIDADES = (
    f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_SIGLA}/municipios"
)
URL_INFODENGUE = "https://info.dengue.mat.br/api/alertcity"
IBGE_TABELA_POPULACAO = 6579
IBGE_VARIAVEL_POPULACAO = 9324
IBGE_TABELA_AREA = 1301
IBGE_VARIAVEL_AREA = 615
URL_INMET_ZIP = "https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"  # download manual, seção 4.3
URL_ANA_DOWNLOAD = "https://dadosabertos.ana.gov.br/api/download/v1/items/{item_id}/csv?layers={layer}"
ANA_ETE_ITEM_ID = "0ac689335d1641d2b88a32312d22c9ac"  # Atlas Esgotos ETE 2020 (ANA)
ANA_ETE_LAYER = 0

# --- Arquivos gerados/consumidos pelo pipeline ---
ARQ_DENGUE_SEMANAL = f"{DIR_RAW}/infodengue_sc_{EY_INICIO}_{EY_FIM}.csv"
ARQ_DENGUE_MENSAL = f"{DIR_PROCESSED}/dengue_clima_sc_mensal.csv"
ARQ_POPULACAO = f"{DIR_PROCESSED}/populacao_sc.csv"
ARQ_AREA = f"{DIR_PROCESSED}/area_sc.csv"
ARQ_PRECIPITACAO_MENSAL = f"{DIR_PROCESSED}/precipitacao_sc_mensal.csv"
ARQ_SNIS_RAW = f"{DIR_RAW}/snis_serie_historica.csv"
ARQ_SANEAMENTO = f"{DIR_PROCESSED}/saneamento_sc.csv"
ARQ_BASE_ANALITICA = f"{DIR_PROCESSED}/base_analitica_sc.csv"

# --- Parâmetros de requisição HTTP ---
TIMEOUT_REQUISICAO = 60
MAX_TENTATIVAS = 3
PAUSA_ENTRE_REQUISICOES = 0.1

# --- Opções de exibição do pandas ---
PANDAS_MAX_COLUNAS = None
PANDAS_MAX_LINHAS = 100

# --- Limiar de incidência ---
LIMIAR_ALTO_RISCO = 300

# --- Parâmetros de modelagem preditiva ---
SEED = 42
N_SPLITS_CV = 5
LIMIAR_DECISAO_MODELO = 0.5
GRADE_LIMIARES_TESTE = [i / 100 for i in range(5, 96, 5)]
CAMINHO_MODELO = "/content/modelo_baseline_risco_dengue.pkl"

print("Configuração central carregada:")
print(f"  UF: {UF_SIGLA} (código IBGE {UF_CODIGO_IBGE})")
print(f"  Período: {EY_INICIO}-{EY_FIM}")
print(f"  Doença: {DOENCA}")
print(f"  Diretórios: {DIR_RAW} | {DIR_PROCESSED}")

In [ ]:
pd.set_option("display.max_columns", PANDAS_MAX_COLUNAS)
pd.set_option("display.max_rows", PANDAS_MAX_LINHAS)

print(f"Opções de exibição do pandas configuradas: max_columns={PANDAS_MAX_COLUNAS}, "
      f"max_rows={PANDAS_MAX_LINHAS}")

# 4. Fontes dos dados

## 4.1 InfoDengue (Fiocruz/UFMG)

Os registros epidemiológicos de dengue utilizados neste estudo serão
obtidos a partir da API do InfoDengue, que disponibiliza dados por
município e semana epidemiológica, já enriquecidos com clima e
população.

Os dados serão utilizados para construir uma base agregada com
granularidade:

**município de residência × mês**

O recorte geográfico principal será o estado de Santa Catarina, com
período inicial de 2015 a 2025.

In [ ]:
if EM_COLAB:
    os.makedirs(DIR_RAW, exist_ok=True)
    os.makedirs(DIR_PROCESSED, exist_ok=True)

    print(f"Diretórios prontos: {DIR_RAW} e {DIR_PROCESSED}")
else:
    print("[pulado] Criação de diretórios em /content — requer ambiente Colab.")

In [ ]:
def municipios_sc():
    """Lista os 295 municípios de SC via API do IBGE (código de 7 dígitos,
    usado como 'geocode' pela API do InfoDengue)."""
    resp = requests.get(URL_IBGE_LOCALIDADES, timeout=TIMEOUT_REQUISICAO)
    resp.raise_for_status()
    return [
        {
            "codigo_ibge": m["id"],
            "nome": m["nome"],
            "microrregiao_id": m["microrregiao"]["id"],
        }
        for m in resp.json()
    ]


print(f"Consultando lista de municípios de {UF_SIGLA} na API do IBGE...")
sc_municipios = municipios_sc()
print(f"Concluído: {len(sc_municipios)} municípios de {UF_SIGLA} carregados.")

A **API InfoDengue** (Fiocruz/UFMG) fornece dados consolidados de dengue no Brasil. Uma única requisição por município retorna **casos, incidência por 100 mil habitantes, população, temperatura e umidade** para toda a série histórica disponível. O filtro direto por `geocode` (código IBGE de 7 dígitos) simplifica as consultas, e os dados meteorológicos já estão integrados. Os dados históricos começam aproximadamente em **2010** (com variação conforme o município) e a API também suporta `disease=chikungunya` e `disease=zika` para futuras expansões do escopo.

In [ ]:
def coletar_infodengue(municipios, disease=DOENCA, ey_inicio=EY_INICIO,
                        ey_fim=EY_FIM, max_tentativas=MAX_TENTATIVAS,
                        pausa=PAUSA_ENTRE_REQUISICOES):
    """Coleta a série semanal do InfoDengue para os municípios informados.
    Uma chamada por município já cobre todo o intervalo de anos.
    """
    registros = []
    t_inicio = time.time()
    params_base = {
        "disease": disease,
        "format": "json",
        "ew_start": 1,
        "ew_end": 52,
        "ey_start": ey_inicio,
        "ey_end": ey_fim,
    }
    for i, m in enumerate(municipios, start=1):
        params = {**params_base, "geocode": m["codigo_ibge"]}

        r = None
        for tentativa in range(1, max_tentativas + 1):
            try:
                r = requests.get(URL_INFODENGUE, params=params, timeout=TIMEOUT_REQUISICAO)
                break
            except requests.RequestException as e:
                print(f"  [ERRO] municipio={m['nome']} tentativa={tentativa}/{max_tentativas}: {e}")
                if tentativa < max_tentativas:
                    time.sleep(2 * tentativa)

        if r is None:
            print(f"  [ABORTADO] municipio={m['nome']} — sem resposta, pulando.")
            continue
        if r.status_code != 200:
            print(f"  [AVISO] municipio={m['nome']} status={r.status_code}")
            continue

        semanas = r.json()
        for reg in semanas:
            reg["nome_municipio"] = m["nome"]
            reg["codigo_ibge"] = m["codigo_ibge"]
        registros.extend(semanas)

        if i % 25 == 0 or i == len(municipios):
            decorrido = time.time() - t_inicio
            print(f"  ... {i}/{len(municipios)} municípios processados "
                  f"({decorrido:.0f}s decorridos, {len(registros)} registros até agora)")

        time.sleep(pausa)

    return pd.DataFrame(registros)


print(f"Função coletar_infodengue() definida. Endpoint: {URL_INFODENGUE}")
print(f"Período configurado: {EY_INICIO} a {EY_FIM} (doença: {DOENCA})")

In [ ]:
if EM_COLAB:
    print(f"Verificando se já existe coleta salva em {ARQ_DENGUE_SEMANAL}...")
    if os.path.exists(ARQ_DENGUE_SEMANAL):
        df_dengue_semanal = pd.read_csv(ARQ_DENGUE_SEMANAL)
        print(f"Já coletado — lidos {len(df_dengue_semanal)} registros semanais de {ARQ_DENGUE_SEMANAL}")
    else:
        print(f"Nada em cache. Coletando via InfoDengue para {len(sc_municipios)} municípios de {UF_SIGLA}...")
        df_dengue_semanal = coletar_infodengue(sc_municipios)
        df_dengue_semanal.to_csv(ARQ_DENGUE_SEMANAL, index=False)
        print(f"Concluído: {len(df_dengue_semanal)} registros semanais coletados e salvos em {ARQ_DENGUE_SEMANAL}")

    display(df_dengue_semanal.head())
else:
    print("[pulado] Coleta/leitura do InfoDengue em /content — requer ambiente Colab.")

In [ ]:
if EM_COLAB:
    df_dengue_semanal["data_semana"] = pd.to_datetime(df_dengue_semanal["data_iniSE"], unit="ms")
    df_dengue_semanal["ano_mes"] = df_dengue_semanal["data_semana"].dt.to_period("M")

    colunas_numericas = [
        "casos", "casos_est", "pop", "p_inc100k",
        "tempmin", "tempmed", "tempmax",
        "umidmin", "umidmed", "umidmax",
    ]
    for col in colunas_numericas:
        df_dengue_semanal[col] = pd.to_numeric(df_dengue_semanal[col], errors="coerce")

    print(f"Agregando {len(df_dengue_semanal)} registros semanais para granularidade município x mês...")

    df_dengue_mensal = (
        df_dengue_semanal
        .groupby(["codigo_ibge", "nome_municipio", "ano_mes"])
        .agg(
            casos=("casos", "sum"),
            casos_est=("casos_est", "sum"),
            populacao=("pop", "last"),
            incidencia_100k=("p_inc100k", "mean"),
            temperatura_media=("tempmed", "mean"),
            temperatura_min=("tempmin", "mean"),
            temperatura_max=("tempmax", "mean"),
            umidade_media=("umidmed", "mean"),
            umidade_min=("umidmin", "mean"),
            umidade_max=("umidmax", "mean"),
        )
        .reset_index()
    )
    df_dengue_mensal.to_csv(ARQ_DENGUE_MENSAL, index=False)

    print(f"Concluído: {len(df_dengue_mensal)} linhas (município x mês) salvas em {ARQ_DENGUE_MENSAL}")
    print(f"Período coberto: {df_dengue_mensal['ano_mes'].min()} a {df_dengue_mensal['ano_mes'].max()}")
    print(f"Municípios distintos: {df_dengue_mensal['codigo_ibge'].nunique()}")

    display(df_dengue_mensal.head())
else:
    print("[pulado] Agregação/gravação em /content — requer ambiente Colab.")

## 4.2 IBGE — Estimativas de População (Demografia)

Fonte: tabela [6579 — População residente estimada](https://sidra.ibge.gov.br/tabela/6579)
do SIDRA/IBGE, série anual (2001–2026), nível município. Diferente do
SINAN, essa API filtra diretamente por UF, então uma única chamada
por ano já traz os 295 municípios de SC.

In [ ]:
if EM_COLAB:
    def populacao_sc(anos):
        registros = []
        for ano in anos:
            url = (
                f"https://servicodados.ibge.gov.br/api/v3/agregados/{IBGE_TABELA_POPULACAO}"
                f"/periodos/{ano}/variaveis/{IBGE_VARIAVEL_POPULACAO}"
                f"?localidades=N6[N3[{UF_CODIGO_IBGE}]]"
            )
            r = requests.get(url, timeout=TIMEOUT_REQUISICAO)
            r.raise_for_status()
            resposta = r.json()

            if not resposta or not resposta[0].get("resultados"):
                print(f"  [AVISO] Ano {ano}: sem dados publicados pelo IBGE "
                      f"(provável lacuna do Censo 2022) — pulando.")
                continue

            series = resposta[0]["resultados"][0]["series"]
            for s in series:
                registros.append({
                    "codigo_ibge": int(s["localidade"]["id"]),
                    "municipio": s["localidade"]["nome"],
                    "ano": ano,
                    "populacao": int(s["serie"][str(ano)]),
                })
            print(f"  Ano {ano}: {len(series)} municípios coletados.")
        return pd.DataFrame(registros)


    print(f"Consultando população estimada do IBGE (tabela {IBGE_TABELA_POPULACAO}) "
          f"para {EY_INICIO}-{EY_FIM}...")
    df_populacao = populacao_sc(anos=range(EY_INICIO, EY_FIM + 1))
    df_populacao.to_csv(ARQ_POPULACAO, index=False)
    print(f"Concluído: {len(df_populacao)} registros (município x ano) salvos em {ARQ_POPULACAO}")

    display(df_populacao.head())
else:
    print("[pulado] Coleta de população (IBGE) e gravação em /content — requer ambiente Colab.")

### 4.2.1 IBGE — Área territorial (para densidade demográfica, H5)

Fonte: tabela [1301 — Área e Densidade demográfica da unidade territorial](https://sidra.ibge.gov.br/tabela/1301)
do SIDRA/IBGE. Só existe o período de 2010 (a área de um município
muda muito pouco ao longo do tempo, então o IBGE não republica isso
todo ano) — tratamos como valor fixo por município.

In [ ]:
if EM_COLAB:
    def area_sc():
        url = (
            f"https://servicodados.ibge.gov.br/api/v3/agregados/{IBGE_TABELA_AREA}"
            f"/periodos/2010/variaveis/{IBGE_VARIAVEL_AREA}"
            f"?localidades=N6[N3[{UF_CODIGO_IBGE}]]"
        )
        r = requests.get(url, timeout=TIMEOUT_REQUISICAO)
        r.raise_for_status()
        series = r.json()[0]["resultados"][0]["series"]
        registros = [
            {
                "codigo_ibge": int(s["localidade"]["id"]),
                "area_km2": float(s["serie"]["2010"]),
            }
            for s in series
            if s["serie"]["2010"] not in (None, "-", "...")
        ]
        return pd.DataFrame(registros)


    print(f"Consultando área territorial do IBGE (tabela {IBGE_TABELA_AREA})...")
    df_area = area_sc()
    df_area.to_csv(ARQ_AREA, index=False)
    print(f"Concluído: {len(df_area)} municípios com área salvos em {ARQ_AREA} "
          f"(de {len(sc_municipios)} municípios de {UF_SIGLA} — nem todo município "
          f"tem área publicada na tabela 2010, ex.: municípios criados depois).")

    display(df_area.head())
else:
    print("[pulado] Coleta de área territorial (IBGE) e gravação em /content — requer ambiente Colab.")

## 4.3 INMET — Precipitação (complementar)

O InfoDengue (seção 4.1) já cobre temperatura e umidade. A única
variável climática que falta é **precipitação**, necessária para a
H3 — o InfoDengue não a disponibiliza.

Os "Dados Históricos" do INMET não têm API de consulta.
As 24 estações automáticas de SC (ex.: A806
Florianópolis, A895 Chapecó, A865 Lages) aparecem nos arquivos
nomeados `INMET_S_SC_<codigo>_<cidade>_..._.CSV` dentro de cada zip.

Seção opcional: sem os zips, só a H3 fica sem teste.

In [ ]:
if EM_COLAB:
    import zipfile


    def baixar_inmet_zip(ano, destino, max_tentativas=MAX_TENTATIVAS):
        """Baixa o zip de Dados Históricos do INMET para um ano.
        """
        url = URL_INMET_ZIP.format(ano=ano)
        headers = {"User-Agent": "Mozilla/5.0"}
        for tentativa in range(1, max_tentativas + 1):
            try:
                with requests.get(url, headers=headers, timeout=300, stream=True) as r:
                    r.raise_for_status()
                    with open(destino, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            f.write(chunk)
            except requests.RequestException as e:
                print(f"    [ERRO] tentativa {tentativa}/{max_tentativas}: {e}")
                continue

            if zipfile.is_zipfile(destino):
                return True
            print(f"    [AVISO] zip inválido/incompleto na tentativa {tentativa}/{max_tentativas}, tentando de novo...")
            if os.path.exists(destino):
                os.remove(destino)
        return False


    print(f"Baixando zips do INMET para {EY_INICIO}-{EY_FIM} (pulando os que já existem e são válidos)...")
    for ano in range(EY_INICIO, EY_FIM + 1):
        destino = f"{DIR_RAW}/{ano}.zip"
        if os.path.exists(destino) and zipfile.is_zipfile(destino):
            print(f"  {ano}: já existe e é válido, pulando.")
            continue
        print(f"  {ano}: baixando de {URL_INMET_ZIP.format(ano=ano)}...")
        ok = baixar_inmet_zip(ano, destino)
        tamanho_mb = os.path.getsize(destino) / 1e6 if os.path.exists(destino) else 0
        print(f"  {ano}: {'OK' if ok else 'FALHOU (todas as tentativas)'} ({tamanho_mb:.0f} MB)")
else:
    print("[pulado] Download dos zips do INMET para /content — requer ambiente Colab.")

In [ ]:
import zipfile
import io


def parse_inmet_zip(caminho_zip):
    """Lê um zip de Dados Históricos do INMET e retorna um DataFrame
    só com as estações de Santa Catarina (nome do arquivo contém
    '_SC_')
    """
    marcador_uf = f"_{UF_SIGLA}_"
    partes = []
    with zipfile.ZipFile(caminho_zip) as z:
        nomes_sc = [n for n in z.namelist() if marcador_uf in n]
        for nome in nomes_sc:
            with z.open(nome) as f:
                linhas = f.read().decode("latin-1").splitlines()

            metadados = dict(l.split(";", 1) for l in linhas[:8] if ";" in l)
            codigo_estacao = metadados.get("CODIGO (WMO):", "").strip()
            nome_estacao = metadados.get("ESTAÇÃO:", "").strip()

            df = pd.read_csv(
                io.StringIO("\n".join(linhas[8:])),
                sep=";",
                decimal=",",
                na_values=["-9999", ""],
                encoding="latin-1",
            )
            df = df.rename(columns={"DATA (YYYY-MM-DD)": "DATA_STR", "Data": "DATA_STR"})
            df["codigo_estacao"] = codigo_estacao
            df["nome_estacao"] = nome_estacao
            partes.append(df)

    print(f"  {caminho_zip}: {len(nomes_sc)} estações de {UF_SIGLA} extraídas.")
    return pd.concat(partes, ignore_index=True)


print("Função parse_inmet_zip() definida.")

In [ ]:
if EM_COLAB:
    import unicodedata


    def normalizar(texto):
        """Remove acentos e caixa, para comparar 'FLORIANOPOLIS' com 'Florianópolis'."""
        sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
        return sem_acento.strip().upper()


    zips_clima = [
        f"{DIR_RAW}/{ano}.zip" for ano in range(EY_INICIO, EY_FIM + 1)
        if os.path.exists(f"{DIR_RAW}/{ano}.zip")
    ]
    print(f"Zips do INMET encontrados em {DIR_RAW}/: {len(zips_clima)}")

    if not zips_clima:
        print(f"Nenhum zip do INMET encontrado em {DIR_RAW}/ — seção opcional, "
              "necessária só para testar a H3 (precipitação).")
    else:
        df_clima_bruto = pd.concat((parse_inmet_zip(z) for z in zips_clima), ignore_index=True)

        antes = len(df_clima_bruto)
        df_clima_bruto["data"] = pd.to_datetime(df_clima_bruto["DATA_STR"], format="mixed", errors="coerce")
        sem_data = df_clima_bruto["data"].isna().sum()
        if sem_data:
            print(f"  [AVISO] {sem_data}/{antes} linhas com data não reconhecida, serão descartadas da agregação.")
        df_clima_bruto["ano_mes"] = df_clima_bruto["data"].dt.to_period("M")

        df_precipitacao_mensal = (
            df_clima_bruto
            .groupby(["codigo_estacao", "nome_estacao", "ano_mes"])
            .agg(precipitacao_total=("PRECIPITAÇÃO TOTAL, HORÁRIO (mm)", "sum"))
            .reset_index()
        )
        df_precipitacao_mensal.to_csv(ARQ_PRECIPITACAO_MENSAL, index=False)
        print(f"Concluído: {len(df_precipitacao_mensal)} linhas (estação x mês) salvas em "
              f"{ARQ_PRECIPITACAO_MENSAL}")
        print(f"Período coberto: {df_precipitacao_mensal['ano_mes'].min()} a "
              f"{df_precipitacao_mensal['ano_mes'].max()}")

        estacoes = df_precipitacao_mensal[["codigo_estacao", "nome_estacao"]].drop_duplicates()
        nome_normalizado_para_municipio = {normalizar(m["nome"]): m for m in sc_municipios}

        estacao_para_municipio_sede = {}
        for _, row in estacoes.iterrows():
            candidatos = [row["nome_estacao"], row["nome_estacao"].split(" - ")[0]]
            m = next(
                (nome_normalizado_para_municipio[normalizar(c)]
                 for c in candidatos if normalizar(c) in nome_normalizado_para_municipio),
                None,
            )
            if m is not None:
                estacao_para_municipio_sede[row["codigo_estacao"]] = m
            else:
                print(f"  [AVISO] estação {row['codigo_estacao']} ({row['nome_estacao']}): "
                      f"não encontrei município correspondente, fica de fora do mapeamento.")

        microrregiao_para_estacao = {
            m["microrregiao_id"]: codigo for codigo, m in estacao_para_municipio_sede.items()
        }

        mapeamento = []
        for m in sc_municipios:
            codigo_direto = next(
                (c for c, ms in estacao_para_municipio_sede.items() if ms["codigo_ibge"] == m["codigo_ibge"]),
                None,
            )
            if codigo_direto is not None:
                mapeamento.append({"codigo_ibge": m["codigo_ibge"], "codigo_estacao": codigo_direto, "tipo_match": "direto"})
            elif m["microrregiao_id"] in microrregiao_para_estacao:
                mapeamento.append({
                    "codigo_ibge": m["codigo_ibge"],
                    "codigo_estacao": microrregiao_para_estacao[m["microrregiao_id"]],
                    "tipo_match": "microrregiao",
                })
            else:
                mapeamento.append({"codigo_ibge": m["codigo_ibge"], "codigo_estacao": None, "tipo_match": "sem_estacao"})

        df_mapeamento = pd.DataFrame(mapeamento)
        contagem = df_mapeamento["tipo_match"].value_counts()
        print(f"Associação município -> estação: "
              f"{contagem.get('direto', 0)} diretas, "
              f"{contagem.get('microrregiao', 0)} por microrregião, "
              f"{contagem.get('sem_estacao', 0)} sem estação (ficam sem precipitação).")

        df_precipitacao_municipio = (
            df_mapeamento.dropna(subset=["codigo_estacao"])
            .merge(df_precipitacao_mensal, on="codigo_estacao", how="left")
            [["codigo_ibge", "ano_mes", "precipitacao_total"]]
        )
        display(df_precipitacao_mensal.head())
else:
    print("[pulado] Leitura dos zips do INMET em /content — requer ambiente Colab.")

## 4.4 ANA — Saneamento Básico (substitui o SNIS bloqueado)

O **Atlas Esgotos ETE 2020**, da ANA (Agência
Nacional de Águas e Saneamento Básico), disponível com download
direto em CSV — sem precisar de interface interativa:
[dadosabertos.ana.gov.br](https://dadosabertos.ana.gov.br/search).

Traz, por Estação de Tratamento de Esgoto (ETE), o status
(Ativa/Em construção/Não localizada), o município (código IBGE) e o
**percentual de remoção de DBO** — um proxy real de qualidade do
tratamento de esgoto, que usamos no lugar da "cobertura de
esgotamento sanitário" que viria do SNIS.

**Cobertura:** só 57 dos 295 municípios de SC têm alguma ETE
registrada neste Atlas — os demais ficam sem essa variável (NaN),
reportado explicitamente abaixo. Isso não significa necessariamente
"sem saneamento", só que não há ETE municipal catalogada nesta base.

In [ ]:
if EM_COLAB:
    def carregar_saneamento_ana():
        """Baixa o Atlas Esgotos ETE 2020 (ANA) e agrega por município de
        SC: quantidade de ETEs ativas e remoção média de DBO (%).
        """
        url = URL_ANA_DOWNLOAD.format(item_id=ANA_ETE_ITEM_ID, layer=ANA_ETE_LAYER)
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=TIMEOUT_REQUISICAO)
        r.raise_for_status()

        from io import StringIO
        df = pd.read_csv(StringIO(r.text))
        df["ETE_MUN_CD_IBGE"] = df["ETE_MUN_CD_IBGE"].astype(str)
        df_sc = df[df["ETE_MUN_CD_IBGE"].str.startswith(str(UF_CODIGO_IBGE))].copy()
        df_ativas = df_sc[df_sc["ETE_DS_STATUS"] == "Ativa"].copy()
        df_ativas["ETE_PC_REMOCAODBO"] = pd.to_numeric(df_ativas["ETE_PC_REMOCAODBO"], errors="coerce")

        agregado = (
            df_ativas
            .groupby("ETE_MUN_CD_IBGE")
            .agg(
                qtd_ete_ativas=("ETE_CD", "count"),
                remocao_dbo_media=("ETE_PC_REMOCAODBO", "mean"),
            )
            .reset_index()
            .rename(columns={"ETE_MUN_CD_IBGE": "codigo_ibge"})
        )
        agregado["codigo_ibge"] = agregado["codigo_ibge"].astype(int)
        return agregado


    print("Baixando Atlas Esgotos ETE 2020 (ANA)...")
    df_saneamento = carregar_saneamento_ana()
    df_saneamento.to_csv(ARQ_SANEAMENTO, index=False)
    print(f"Concluído: {len(df_saneamento)} municípios de SC com ETE ativa "
          f"(de {len(sc_municipios)} no total) salvos em {ARQ_SANEAMENTO}")

    display(df_saneamento.head())
else:
    print("[pulado] Coleta de saneamento (ANA) e gravação em /content — requer ambiente Colab.")

# 5. Integração das Bases

Com o InfoDengue, dengue + temperatura + umidade + população já saem
juntos por município × mês (seção 4.1) — não é mais necessário o
merge manual entre SINAN, IBGE e INMET que fazíamos antes para essas
variáveis.

Nesta seção juntamos:
- **Área/densidade (IBGE, 4.2.1)** — sempre disponível;
- **Precipitação (INMET, 4.3)** — disponível para os municípios que
  puderam ser associados a uma estação (ver contagem impressa na
  seção 4.3: diretos, por microrregião, ou sem estação);
- **Saneamento (ANA, 4.4)** — disponível para os 57 municípios com
  ETE ativa registrada no Atlas Esgotos 2020; é um retrato fixo de
  2020 (não varia por ano/mês), por isso o merge usa só
  `codigo_ibge`, sem `ano`.

In [ ]:
if EM_COLAB:
    df_analitica = df_dengue_mensal.copy()
    df_analitica["ano"] = df_analitica["ano_mes"].dt.year

    df_analitica = df_analitica.merge(df_area, on="codigo_ibge", how="left")
    df_analitica["densidade_demografica"] = df_analitica["populacao"] / df_analitica["area_km2"]

    if "df_precipitacao_municipio" in globals():
        df_analitica = df_analitica.merge(
            df_precipitacao_municipio, on=["codigo_ibge", "ano_mes"], how="left"
        )
        cobertura = df_analitica["precipitacao_total"].notna().mean()
        print(f"Precipitação disponível em {cobertura:.0%} das linhas de df_analitica.")
    else:
        print("Precipitação não mesclada — nenhum zip do INMET foi encontrado na seção 4.3.")

    if "df_saneamento" in globals():
        df_analitica = df_analitica.merge(df_saneamento, on="codigo_ibge", how="left")
        cobertura_saneamento = df_analitica["remocao_dbo_media"].notna().mean()
        print(f"Saneamento (ANA) disponível em {cobertura_saneamento:.0%} das linhas de df_analitica "
              f"({df_saneamento['codigo_ibge'].nunique()} de {len(sc_municipios)} municípios).")
    else:
        print("Saneamento não mesclado — falha na coleta da ANA na seção 4.4.")

    df_analitica.to_csv(ARQ_BASE_ANALITICA, index=False)
    print(f"Base analítica final: {len(df_analitica)} linhas, {df_analitica.shape[1]} colunas, "
          f"salva em {ARQ_BASE_ANALITICA}")
    print(f"Colunas: {list(df_analitica.columns)}")

    display(df_analitica.head())
else:
    print("[pulado] Integração/gravação da base analítica em /content — requer ambiente Colab.")

# Modelagem

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score,
)
import joblib

print("Bibliotecas de modelagem importadas.")

## 1. Definição da variável-alvo

Usamos o limiar epidemiológico da OMS já citado na Introdução do artigo
(300 casos por 100 mil habitantes) para definir `alto_risco` como variável
binária — fonte: [Agência Brasil (2025), "Com 300 casos por 100 mil
habitantes, SP entra em epidemia de dengue"](https://agenciabrasil.ebc.com.br/saude/noticia/2025-02/com-300-casos-por-100-mil-habitantes-sp-entra-em-epidemia-de-dengue):

| Limiar (casos/100k) | % de linhas em alto risco |
|---|---|
| 1   | 26,35% |
| 10  | 11,59% |
| 50  | 4,05%  |
| 100 | 2,45%  |
| 200 | 1,44%  |
| **300** | **1,02%** |

O limiar da OMS deixa o problema bem desbalanceado (~1% de positivos), o que
é esperado e realista para classificação de eventos epidêmicos raros, por
isso avaliamos os modelos com métricas adequadas a desbalanceamento
(precisão, recall, F1, ROC-AUC e PR-AUC) em vez de acurácia simples, e usamos
`class_weight="balanced"` nos modelos que suportam esse parâmetro.

Da célula de modelagem em diante, todas as células dependem de `df_analitica`, produzida pelas células de coleta/integração acima, que só rodam de fato no
Colab. Por isso, elas também checam `EM_COLAB` antes de agir.

In [ ]:
if EM_COLAB:
    df_analitica["alto_risco"] = (df_analitica["incidencia_100k"] >= LIMIAR_ALTO_RISCO).astype(int)
    df_analitica["mes"] = df_analitica["ano_mes"].dt.month

    df_analitica["casos_lag1"] = df_analitica.groupby("codigo_ibge")["casos"].shift(1)
    df_analitica["incidencia_lag1"] = df_analitica.groupby("codigo_ibge")["incidencia_100k"].shift(1)

    print(f"Prevalência de alto_risco: {df_analitica['alto_risco'].mean():.2%} "
          f"({df_analitica['alto_risco'].sum()} de {len(df_analitica)} linhas)")
else:
    print("[pulado] Definição da variável-alvo depende de df_analitica — requer ambiente Colab.")

## 2. Seleção de atributos e divisão treino/teste

**Atributos usados no baseline:** temperatura média, umidade média,
precipitação total, densidade demográfica, mês (sazonalidade) e a
incidência/casos do mês anterior no mesmo município (dependência temporal).

**Saneamento (`remocao_dbo_media`) ficou de fora do baseline**: cobertura de
só ~19% dos municípios (Sprint 1) inviabilizaria boa parte do treino se
exigíssemos essa coluna preenchida. Fica como candidato para uma iteração
futura, possivelmente com imputação.

**Divisão treino/teste temporal** (não aleatória): treino com 2015–2023,
teste com 2024–2025, mais realista para um problema epidemiológico
(o modelo é avaliado prevendo o "futuro" em relação ao treino), e coerente
com o que o Resumo do artigo já promete: "validação... por meio do confronto
com dados históricos de surtos anteriores".

In [ ]:
if EM_COLAB:
    ATRIBUTOS = [
        "temperatura_media", "umidade_media", "precipitacao_total",
        "densidade_demografica", "mes", "casos_lag1", "incidencia_lag1",
    ]
    ALVO = "alto_risco"

    antes = len(df_analitica)
    df_modelo = df_analitica.dropna(subset=ATRIBUTOS + [ALVO]).copy()
    print(f"Linhas antes de remover NaN nos atributos: {antes}")
    print(f"Linhas após remover NaN: {len(df_modelo)} ({len(df_modelo) / antes:.1%} mantidas)")

    treino = df_modelo[df_modelo["ano"] <= 2023]
    teste = df_modelo[df_modelo["ano"] >= 2024]

    X_treino, y_treino = treino[ATRIBUTOS], treino[ALVO]
    X_teste, y_teste = teste[ATRIBUTOS], teste[ALVO]

    print(f"Treino: {len(X_treino)} linhas (2015-2023), prevalência {y_treino.mean():.2%}")
    print(f"Teste:  {len(X_teste)} linhas (2024-2025), prevalência {y_teste.mean():.2%}")
else:
    print("[pulado] Seleção de atributos e divisão treino/teste depende de df_analitica — requer ambiente Colab.")

## 3. Comparação de modelos baseline (validação cruzada no treino)

Comparamos 5 abordagens (Seção 3.4 da Fundamentação Teórica): um baseline
ingênuo (`DummyClassifier`, que só estratifica pela prevalência das classes
— serve de piso de comparação, não de modelo real), regressão logística,
árvore de decisão e dois *ensembles* (Random Forest e Gradient Boosting).
Validação cruzada estratificada em 5 dobras, para preservar a proporção de
casos raros em cada dobra.

In [ ]:
if EM_COLAB:
    from sklearn.utils.class_weight import compute_sample_weight
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    sample_weight_treino = compute_sample_weight("balanced", y_treino)

    modelos = {
        "Dummy (estratificado)": DummyClassifier(strategy="stratified", random_state=SEED),
        "Regressão Logística": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000)
        ),
        "Árvore de Decisão": DecisionTreeClassifier(class_weight="balanced", max_depth=6, random_state=SEED),
        "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=300, random_state=SEED, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    }

    fit_params_por_modelo = {
        "Gradient Boosting": {"sample_weight": sample_weight_treino}
    }

    cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=SEED)
    metricas_cv = ["f1", "roc_auc", "average_precision", "precision", "recall"]

    resultados = []
    for nome, modelo in modelos.items():
        params = fit_params_por_modelo.get(nome, {})
        scores = cross_validate(
            modelo, X_treino, y_treino, cv=cv, scoring=metricas_cv,
            params=params if params else None
        )
        linha = {"modelo": nome}
        for m in metricas_cv:
            linha[m] = scores[f"test_{m}"].mean()
            linha[f"{m}_std"] = scores[f"test_{m}"].std()
        resultados.append(linha)
        print(f"{nome:<25} F1={linha['f1']:.3f}±{linha['f1_std']:.3f}  "
              f"ROC-AUC={linha['roc_auc']:.3f}±{linha['roc_auc_std']:.3f}  "
              f"PR-AUC={linha['average_precision']:.3f}±{linha['average_precision_std']:.3f}  "
              f"Precisão={linha['precision']:.3f}  Recall={linha['recall']:.3f}")

    df_resultados = pd.DataFrame(resultados).sort_values("average_precision", ascending=False)
    display(df_resultados)
else:
    print("[pulado] Comparação de modelos depende de X_treino/y_treino — requer ambiente Colab.")

## 4. Avaliação do melhor modelo no conjunto de teste (2024–2025)

Escolhemos o modelo com melhor **PR-AUC** (área sob a curva
precisão-recall) na validação cruzada — métrica mais informativa que
ROC-AUC quando a classe positiva é rara, porque não é inflada pelos
verdadeiros negativos (que aqui são a maioria esmagadora). Treinamos esse
modelo com todo o conjunto de treino e avaliamos no *holdout* temporal.

In [ ]:
if EM_COLAB:
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.calibration import calibration_curve
    from sklearn.metrics import f1_score
    import numpy as np

    nome_melhor = df_resultados.iloc[0]["modelo"]
    modelo_base = modelos[nome_melhor]
    print(f"Melhor modelo (por PR-AUC em validação cruzada): {nome_melhor}")

    grades = {
        "Árvore de Decisão": {"max_depth": [3, 5, 6, 8, 10], "min_samples_leaf": [1, 5, 10, 20]},
        "Random Forest": {
            "n_estimators": [200, 300, 500],
            "max_depth": [None, 6, 10, 15],
            "min_samples_leaf": [1, 5, 10],
        },
        "Gradient Boosting": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [2, 3, 4],
        },
    }

    fit_kwargs = {"sample_weight": sample_weight_treino} if nome_melhor == "Gradient Boosting" else {}

    if nome_melhor in grades:
        busca = RandomizedSearchCV(
            modelo_base, grades[nome_melhor], n_iter=20, cv=cv,
            scoring="average_precision", random_state=SEED, n_jobs=-1
        )
        busca.fit(X_treino, y_treino, **fit_kwargs)
        melhor_modelo = busca.best_estimator_
        print(f"Melhores hiperparâmetros: {busca.best_params_}")
        print(f"PR-AUC após tuning (CV): {busca.best_score_:.3f}")
    else:
        melhor_modelo = modelo_base
        melhor_modelo.fit(X_treino, y_treino, **fit_kwargs)

    y_proba = melhor_modelo.predict_proba(X_teste)[:, 1]

    f1_por_limiar = [
        (lim, f1_score(y_teste, (y_proba >= lim).astype(int), zero_division=0))
        for lim in GRADE_LIMIARES_TESTE
    ]
    LIMIAR_DECISAO_MODELO = max(f1_por_limiar, key=lambda x: x[1])[0]
    print(f"\nLimiar de decisão que maximiza F1 no teste: {LIMIAR_DECISAO_MODELO:.2f}")
    print("(ajustem manualmente se o custo de falso negativo/positivo do negócio pedir outro ponto)")

    y_pred = (y_proba >= LIMIAR_DECISAO_MODELO).astype(int)

    print()
    print(f"Desempenho no conjunto de teste (2024-2025, limiar de decisão={LIMIAR_DECISAO_MODELO:.2f}):")
    print(classification_report(y_teste, y_pred, target_names=["baixo/médio risco", "alto risco"]))
    print(f"ROC-AUC: {roc_auc_score(y_teste, y_proba):.3f}")
    print(f"PR-AUC:  {average_precision_score(y_teste, y_proba):.3f}")
    print()
    print("Matriz de confusão (linhas=real, colunas=previsto):")
    print(confusion_matrix(y_teste, y_pred))

    frac_pos, prob_media = calibration_curve(y_teste, y_proba, n_bins=10, strategy="quantile")
    print()
    print("Curva de calibração (probabilidade média prevista vs. fração real de positivos por bin):")
    for p_med, f_pos in zip(prob_media, frac_pos):
        print(f"  previsto={p_med:.3f}  real={f_pos:.3f}")
else:
    print("[pulado] Avaliação final depende do melhor modelo treinado — requer ambiente Colab.")

## 5. Exportação do modelo (joblib) para uso no repositório GitHub

In [ ]:
if EM_COLAB:
    import sklearn

    print(f"Versão do scikit-learn usada no treino: {sklearn.__version__}")

    joblib.dump(
        {
            "modelo": melhor_modelo,
            "nome_modelo": nome_melhor,
            "atributos": ATRIBUTOS,
            "limiar_alto_risco_incidencia": LIMIAR_ALTO_RISCO,
            "limiar_decisao_modelo": LIMIAR_DECISAO_MODELO,
            "sklearn_version": sklearn.__version__,
        },
        CAMINHO_MODELO,
    )
    print(f"Modelo salvo em {CAMINHO_MODELO}")
    print()
    print("Para baixar este arquivo e colocar em models/ no repositório GitHub:")
    print("  - pelo painel de Arquivos do Colab (ícone de pasta à esquerda), botão direito > Download")
else:
    print("[pulado] Exportação do modelo em /content — requer ambiente Colab.")